# Birds v2 — Improved Pipeline

Improvements over v1:
- **Richer features**: raw pixels + per-row/column statistics (explicit time-frequency structure)
- **MLPClassifier**: sklearn neural net, much stronger than PCA+SVM for image data
- **XGBoost & LightGBM**: gradient boosting on engineered features
- **Ensemble voting**: majority vote across top models

**AI note**: Claude Code (claude-sonnet-4-6) scaffolded this notebook after reading competition context, exploring the data directory, and benchmarking available packages. All modelling decisions were reviewed by the student.

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import scipy.ndimage as ndi
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import lightgbm as lgb

BASE      = Path('All_about_Birds/All_about_Birds')
TRAIN_DIR = BASE / 'train'
TEST_DIR  = BASE / 'test'
SAMPLE    = Path('sample_submission.csv')

IMG_H, IMG_W = 32, 64  # height x width

print('Classes:', [d.name for d in sorted(TRAIN_DIR.iterdir()) if d.is_dir()])

Classes: ['Antbird', 'Bird of prey', 'Flycatcher', 'Ground bird', 'Nocturnal bird', 'Other non-passerine bird', 'Parrot', 'Tanager', 'Water-associated bird']


## 1. Feature Engineering

Each 64×32 spectrogram image encodes **frequency (rows) × time (columns)**.
We extract:
- Raw flattened pixels (2048 dims)
- Per-row mean & std → frequency envelope (64 dims)
- Per-col mean & std → temporal envelope (64 dims)
- 4×4 block means → coarse spatial grid (16 dims)
- Global stats: mean, std, skew, percentiles

Total: ~2200 features.

In [2]:
from scipy.stats import skew, kurtosis

def extract_features(img_array):
    """img_array: float32 H×W in [0,1]"""
    flat   = img_array.flatten()

    # Per-row and per-column stats
    row_mean = img_array.mean(axis=1)   # (H,)
    row_std  = img_array.std(axis=1)
    col_mean = img_array.mean(axis=0)   # (W,)
    col_std  = img_array.std(axis=0)

    # 4×4 block means (coarse grid)
    bh, bw = IMG_H // 4, IMG_W // 4
    blocks = []
    for r in range(4):
        for c in range(4):
            blocks.append(img_array[r*bh:(r+1)*bh, c*bw:(c+1)*bw].mean())
    blocks = np.array(blocks)

    # Global statistics
    glob = np.array([
        flat.mean(), flat.std(),
        skew(flat), kurtosis(flat),
        np.percentile(flat, 25),
        np.percentile(flat, 75),
        flat.max(), flat.min(),
    ])

    return np.concatenate([flat, row_mean, row_std, col_mean, col_std, blocks, glob])

# Sanity check
dummy = np.random.rand(IMG_H, IMG_W).astype(np.float32)
print('Feature vector length:', extract_features(dummy).shape[0])

Feature vector length: 2264


## 2. Load All Data

In [4]:
def load_dir(directory, label=None):
    feats, labels = [], []
    for p in sorted(directory.glob('*.png')):
        img = np.array(Image.open(p).convert('L'), dtype=np.float32) / 255.0
        feats.append(extract_features(img))
        if label is not None:
            labels.append(label)
    return feats, labels

print('Loading training images...')
t0 = time.time()
X_list, y_list = [], []
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    f, l = load_dir(cls_dir, cls_dir.name)
    X_list.extend(f); y_list.extend(l)
    print(f'  {cls_dir.name:35s} {len(f):5d}')

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list)
print(f'\nTraining: X={X.shape}  ({time.time()-t0:.1f}s)')

print('\nLoading test images...')
t0 = time.time()
sample_sub     = pd.read_csv(SAMPLE)
test_filenames = sample_sub['file_name'].tolist()
X_test_list = []
for fname in test_filenames:
    img = np.array(Image.open(TEST_DIR / fname).convert('L'), dtype=np.float32) / 255.0
    X_test_list.append(extract_features(img))
X_test = np.array(X_test_list, dtype=np.float32)
print(f'Test:     X_test={X_test.shape}  ({time.time()-t0:.1f}s)')

Loading training images...
  Antbird                              1586
  Bird of prey                         1959
  Flycatcher                           5143
  Ground bird                           756
  Nocturnal bird                       2688
  Other non-passerine bird             4144
  Parrot                               1175
  Tanager                              1750
  Water-associated bird                2628

Training: X=(21829, 2264)  (478.1s)

Loading test images...
Test:     X_test=(5454, 2264)  (132.7s)


## 3. Encode & Split

In [5]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Classes:', list(le.classes_))

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y_enc, test_size=0.15, random_state=42, stratify=y_enc
)
print(f'Train: {X_tr.shape}  Val: {X_val.shape}')

scaler = StandardScaler()
X_tr_s   = scaler.fit_transform(X_tr)
X_val_s  = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

Classes: [np.str_('Antbird'), np.str_('Bird of prey'), np.str_('Flycatcher'), np.str_('Ground bird'), np.str_('Nocturnal bird'), np.str_('Other non-passerine bird'), np.str_('Parrot'), np.str_('Tanager'), np.str_('Water-associated bird')]
Train: (18554, 2264)  Val: (3275, 2264)


## 4. Model A — MLPClassifier (Neural Network)

Two hidden layers; `early_stopping` prevents overfitting.

In [6]:
print('Training MLP...')
t0 = time.time()

mlp = MLPClassifier(
    hidden_layer_sizes=(1024, 512, 256),
    activation='relu',
    solver='adam',
    alpha=1e-4,          # L2 regularisation
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=200,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    random_state=42,
    verbose=True,
)
mlp.fit(X_tr_s, y_tr)

mlp_val_acc = accuracy_score(y_val, mlp.predict(X_val_s))
print(f'\nMLP val accuracy: {mlp_val_acc:.4f}  ({time.time()-t0:.1f}s)')

Training MLP...
Iteration 1, loss = 2.39449159
Validation score: 0.316272
Iteration 2, loss = 1.87543229
Validation score: 0.320043
Iteration 3, loss = 1.76265101
Validation score: 0.335668
Iteration 4, loss = 1.70017213
Validation score: 0.336746
Iteration 5, loss = 1.64525370
Validation score: 0.353987
Iteration 6, loss = 1.56606972
Validation score: 0.344828
Iteration 7, loss = 1.49858626
Validation score: 0.349138
Iteration 8, loss = 1.43195106
Validation score: 0.362608
Iteration 9, loss = 1.36329578
Validation score: 0.362069
Iteration 10, loss = 1.30878235
Validation score: 0.334052
Iteration 11, loss = 1.26220463
Validation score: 0.364224
Iteration 12, loss = 1.18062453
Validation score: 0.340517
Iteration 13, loss = 1.12211992
Validation score: 0.362608
Iteration 14, loss = 1.07371655
Validation score: 0.364224
Iteration 15, loss = 1.04216437
Validation score: 0.335668
Iteration 16, loss = 0.97579078
Validation score: 0.367996
Iteration 17, loss = 0.93161997
Validation score:

## 5. Model B — XGBoost

In [7]:
print('Training XGBoost...')
t0 = time.time()

xgb_clf = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    early_stopping_rounds=20,
    n_jobs=-1,
    random_state=42,
    device='cpu',
)
xgb_clf.fit(
    X_tr_s, y_tr,
    eval_set=[(X_val_s, y_val)],
    verbose=50,
)

xgb_val_acc = accuracy_score(y_val, xgb_clf.predict(X_val_s))
print(f'\nXGBoost val accuracy: {xgb_val_acc:.4f}  ({time.time()-t0:.1f}s)')

Training XGBoost...
[0]	validation_0-mlogloss:2.00832
[50]	validation_0-mlogloss:1.63393
[100]	validation_0-mlogloss:1.61169
[149]	validation_0-mlogloss:1.60598

XGBoost val accuracy: 0.4333  (361.6s)


## 6. Model C — LightGBM

In [8]:
print('Training LightGBM...')
t0 = time.time()

lgb_clf = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)
lgb_clf.fit(
    X_tr_s, y_tr,
    eval_set=[(X_val_s, y_val)],
    callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(50)],
)

lgb_val_acc = accuracy_score(y_val, lgb_clf.predict(X_val_s))
print(f'\nLightGBM val accuracy: {lgb_val_acc:.4f}  ({time.time()-t0:.1f}s)')

Training LightGBM...
[50]	valid_0's multi_logloss: 1.62457
[100]	valid_0's multi_logloss: 1.60349

LightGBM val accuracy: 0.4342  (89.2s)


## 7. Model D — SVM-RBF on PCA features

PCA + SVM is cheap and often complementary to tree/MLP models in an ensemble.

In [9]:
print('Training SVM-RBF with PCA(300)...')
t0 = time.time()

pca = PCA(n_components=300, random_state=42)
X_tr_pca   = pca.fit_transform(X_tr_s)
X_val_pca  = pca.transform(X_val_s)
X_test_pca = pca.transform(X_test_s)
print(f'Explained variance: {pca.explained_variance_ratio_.sum():.3f}')

svm = SVC(kernel='rbf', C=20, gamma='scale', probability=True, random_state=42)
svm.fit(X_tr_pca, y_tr)

svm_val_acc = accuracy_score(y_val, svm.predict(X_val_pca))
print(f'SVM val accuracy: {svm_val_acc:.4f}  ({time.time()-t0:.1f}s)')

Training SVM-RBF with PCA(300)...
Explained variance: 0.906
SVM val accuracy: 0.3875  (681.5s)


## 8. Summary & Ensemble

In [10]:
results = {
    'MLP':      mlp_val_acc,
    'XGBoost':  xgb_val_acc,
    'LightGBM': lgb_val_acc,
    'SVM-RBF':  svm_val_acc,
}
print('=== Individual model val accuracy ===')
for name, acc in sorted(results.items(), key=lambda x: -x[1]):
    print(f'  {name:<12} {acc:.4f}')

# Ensemble: majority vote using predicted class labels
# SVM and MLP operate on PCA / scaled space; XGB & LGB operate on scaled space
preds_val = {
    'mlp':  mlp.predict(X_val_s),
    'xgb':  xgb_clf.predict(X_val_s),
    'lgb':  lgb_clf.predict(X_val_s),
    'svm':  svm.predict(X_val_pca),
}

# Stack predictions and take majority vote
stacked_val = np.stack(list(preds_val.values()), axis=1)  # (N, 4)
from scipy.stats import mode
ensemble_val = mode(stacked_val, axis=1).mode.flatten()
ens_acc = accuracy_score(y_val, ensemble_val)
print(f'\n  Ensemble (vote)  {ens_acc:.4f}')

=== Individual model val accuracy ===
  LightGBM     0.4342
  XGBoost      0.4333
  SVM-RBF      0.3875
  MLP          0.3847

  Ensemble (vote)  0.4376


## 9. Retrain Best Strategy on Full Data & Generate Submission

In [11]:
# Determine whether to use ensemble or single best model
best_single = max(results, key=results.get)
use_ensemble = ens_acc > results[best_single]
print(f'Best single: {best_single} ({results[best_single]:.4f})')
print(f'Ensemble:    {ens_acc:.4f}')
print(f'Using: {"Ensemble" if use_ensemble else best_single}')

Best single: LightGBM (0.4342)
Ensemble:    0.4376
Using: Ensemble


In [12]:
# Retrain on full data
print('Retraining on full dataset...')
t0 = time.time()

scaler2 = StandardScaler()
X_all_s   = scaler2.fit_transform(X)
X_test_s2 = scaler2.transform(X_test)

# MLP
mlp2 = MLPClassifier(
    hidden_layer_sizes=(1024, 512, 256), activation='relu', solver='adam',
    alpha=1e-4, batch_size=256, learning_rate_init=1e-3,
    max_iter=200, early_stopping=True, validation_fraction=0.05,
    n_iter_no_change=15, random_state=42, verbose=False,
)
mlp2.fit(X_all_s, y_enc)
print(f'  MLP done ({time.time()-t0:.1f}s)')

# XGBoost
t1 = time.time()
xgb2 = xgb.XGBClassifier(
    n_estimators=xgb_clf.best_iteration + 1 if hasattr(xgb_clf, 'best_iteration') else 300,
    max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', n_jobs=-1, random_state=42, device='cpu',
)
xgb2.fit(X_all_s, y_enc, verbose=False)
print(f'  XGB done ({time.time()-t1:.1f}s)')

# LightGBM
t1 = time.time()
lgb2 = lgb.LGBMClassifier(
    n_estimators=lgb_clf.best_iteration_ if hasattr(lgb_clf, 'best_iteration_') and lgb_clf.best_iteration_ else 300,
    max_depth=7, learning_rate=0.05, num_leaves=63,
    subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42, verbose=-1,
)
lgb2.fit(X_all_s, y_enc)
print(f'  LGB done ({time.time()-t1:.1f}s)')

# SVM on PCA
t1 = time.time()
pca2 = PCA(n_components=300, random_state=42)
X_all_pca   = pca2.fit_transform(X_all_s)
X_test_pca2 = pca2.transform(X_test_s2)
svm2 = SVC(kernel='rbf', C=20, gamma='scale', probability=True, random_state=42)
svm2.fit(X_all_pca, y_enc)
print(f'  SVM done ({time.time()-t1:.1f}s)')

print(f'Total retraining: {time.time()-t0:.1f}s')

Retraining on full dataset...
  MLP done (227.0s)
  XGB done (280.3s)
  LGB done (87.2s)
  SVM done (788.1s)
Total retraining: 1382.6s


In [13]:
# Generate test predictions
preds_test = np.stack([
    mlp2.predict(X_test_s2),
    xgb2.predict(X_test_s2),
    lgb2.predict(X_test_s2),
    svm2.predict(X_test_pca2),
], axis=1)

if use_ensemble:
    final_enc = mode(preds_test, axis=1).mode.flatten()
else:
    model_map = {'MLP': mlp2, 'XGBoost': xgb2, 'LightGBM': lgb2}
    if best_single == 'SVM-RBF':
        final_enc = svm2.predict(X_test_pca2)
    else:
        final_enc = model_map[best_single].predict(X_test_s2)

final_labels = le.inverse_transform(final_enc)

submission = pd.DataFrame({'file_name': test_filenames, 'label': final_labels})
submission.to_csv('submission_v2.csv', index=False)
print(f'Saved submission_v2.csv ({len(submission)} rows)')
print('Distribution:')
print(pd.Series(final_labels).value_counts().sort_index())

Saved submission_v2.csv (5454 rows)
Distribution:
Antbird                      294
Bird of prey                 169
Flycatcher                  2207
Ground bird                   70
Nocturnal bird               649
Other non-passerine bird    1167
Parrot                       169
Tanager                      219
Water-associated bird        510
Name: count, dtype: int64


## 10. Per-class Report on Validation Set

In [14]:
# Best performing individual on val
print(f'=== Classification report: {best_single} ===')
model_preds = {
    'MLP': preds_val['mlp'],
    'XGBoost': preds_val['xgb'],
    'LightGBM': preds_val['lgb'],
    'SVM-RBF': preds_val['svm'],
}
print(classification_report(y_val, model_preds[best_single], target_names=le.classes_))

if use_ensemble:
    print('=== Classification report: Ensemble ===')
    print(classification_report(y_val, ensemble_val, target_names=le.classes_))

=== Classification report: LightGBM ===
                          precision    recall  f1-score   support

                 Antbird       0.51      0.29      0.37       238
            Bird of prey       0.34      0.09      0.14       294
              Flycatcher       0.47      0.72      0.56       772
             Ground bird       0.68      0.20      0.31       113
          Nocturnal bird       0.41      0.40      0.41       403
Other non-passerine bird       0.39      0.51      0.44       622
                  Parrot       0.49      0.26      0.34       176
                 Tanager       0.55      0.37      0.44       263
   Water-associated bird       0.35      0.31      0.33       394

                accuracy                           0.43      3275
               macro avg       0.47      0.35      0.37      3275
            weighted avg       0.44      0.43      0.41      3275

=== Classification report: Ensemble ===
                          precision    recall  f1-score   s